In [1]:
import numpy as np 
import pandas as pd

movies=pd.read_csv('final.csv')
movies[movies.duplicated(subset='id', keep=False)]
movies.drop_duplicates(subset='id', keep='first', inplace=True)

In [2]:

from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

def stem(text):
    l=[]
    for i in text.split():
        l.append(ps.stem(i))
    string=" ".join(l)
    return string

In [3]:
director=movies[['id','director']]

director = director.copy()
director['director'] = director['director'].fillna('unknown_director')

from sklearn.feature_extraction.text import CountVectorizer

cv_director = CountVectorizer(stop_words='english')
director_vectors = cv_director.fit_transform(director['director'])
director_vectors = director_vectors.tolil()   #: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
unknown_index=cv_director.vocabulary_.get('unknown_director')

if unknown_index is not None:
    director_vectors=director_vectors.copy()
    director_vectors[:,unknown_index]=0
director_vectors = director_vectors.tocsr()       # Convert back to CSR for fast computations


In [4]:

director_count=director_vectors.toarray().sum(axis=0)
director_names=cv_director.get_feature_names_out()
director_freq=list(zip(director_names,director_count))

import os
os.makedirs('director',exist_ok=True)

top_director=sorted(director_freq, key= lambda x:x[1],reverse=True)[:50]
df_top = pd.DataFrame(top_director, columns=['director', 'count'])
df_top['rank'] = df_top.index
df_top[['rank', 'director']].to_csv('director/top_100_director.csv', index=False)

In [5]:
top_director=df_top['director'].tolist()

cv_top=CountVectorizer(vocabulary=top_director)
top_ind=cv_top.fit_transform(director['director'])

top_dense=top_ind.toarray()
np.savez_compressed('director/movie_director_mapping.npz',top_dense)


#To read the array
data = np.load('director/movie_director_mapping.npz')
arr = data['arr_0'] 

print("Shape of array:", arr.shape)
print("First row (movie 0):", arr[99])

Shape of array: (3000, 50)
First row (movie 0): [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [6]:
loaded = np.load('final/final_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])
loaded = np.load('final/final_similarity.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])

[2, 4, 10, 54, 72, 86, 112, 119, 150, 151, 190, 223, 244, 250, 288, 320, 345, 364, 377, 426, 572, 589, 734, 751, 1055, 1451, 1917, 2217, 12, 953, 6, 7, 30, 97, 108, 183, 219, 331, 380, 427, 457, 463, 514, 528, 840, 961, 1678, 1956, 2067, 2500, 2668]
[np.int64(9), np.int64(6), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]


In [7]:
top_k = 300
top_similar_movies = []

final_similarity = np.load('final/final_similarity.npz', allow_pickle=True)['arr_0']
final_movies = np.load('final/final_movies.npz', allow_pickle=True)['arr_0']

updated_similarities = []
updated_movies = []

os.makedirs('director', exist_ok=True)


for i in range(top_dense.shape[0]):
    cast_i=top_dense[i]
    similarities=[]

    existing_movies=list(final_movies[i])
    existing_sims=list(final_similarity[i])
    sim_dict={movie_id:sim for movie_id,sim in zip(existing_movies,existing_sims)}
    

    for j in range (top_dense.shape[0]):
        if i==j:
            continue

        cast_j = top_dense[j]  # Replace this with `director_dense[j]` if available
        sim = np.sum(np.minimum(cast_i, cast_j))
        
        if sim <= 0:
            continue

        if j in sim_dict:
            sim_dict[j]+=sim
        elif len(sim_dict)<top_k:
            sim_dict[j]=sim
            
    sorted_sims = sorted(sim_dict.items(), key=lambda x: x[1], reverse=True)[:top_k]
    updated_movies.append([j for j, _ in sorted_sims])
    updated_similarities.append([sim for _, sim in sorted_sims])
            
    
    # Pad with -1 if fewer than top_k
    padded = [j for j, _ in sorted_sims] + [-1] * (top_k - len(sorted_sims))
    top_similar_movies.append(padded)
np.savez_compressed('director/top_300_similar_movies.npz', top_similar_movies)
np.savez_compressed('final/final_similarity.npz', np.array(updated_similarities, dtype=object))
np.savez_compressed('final/final_movies.npz', np.array(updated_movies, dtype=object))

In [8]:
loaded = np.load('director/top_300_similar_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[99]
print(first_row)


[  59   69   76   94  214  297  488  556  588  754 1123 1779 1844 2251
 2362 2795 2850   27  171  900 1066 1791   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1   -1
   -1 

In [9]:
loaded = np.load('final/final_movies.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])
loaded = np.load('final/final_similarity.npz', allow_pickle=True)
arr = loaded['arr_0']
first_row = arr[1]
print(first_row[:151])

[2, 4, 10, 54, 72, 86, 112, 119, 150, 151, 190, 223, 244, 250, 288, 320, 345, 364, 377, 426, 572, 589, 734, 751, 1055, 1451, 1917, 2217, 12, 953, 840, 6, 7, 30, 97, 108, 183, 219, 331, 380, 427, 457, 463, 514, 528, 961, 1678, 1956, 2067, 2500, 2668]
[np.int64(10), np.int64(7), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(3), np.int64(2), np.int64(2), np.int64(2), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1)]
